# 02 Data Cleaning & Preprocessing

This notebook focuses on cleaning the merged Steam games dataset. We will address missing values, incorrect data types, and prepare the features for analysis.

## Objectives:
- Load the merged dataset.
- Handle missing values and duplicates.
- Standardize data types (Dates, Numerics).
- Feature engineering (Extracting AppIDs).
- Advanced imputation for categorical and numeric gaps.
- Save the cleaned dataset to `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
import os
import re

# Load the merged data
merged_file = '../data/merged/merged_game_data.csv'
df = pd.read_csv(merged_file)
print(f"Initial shape: {df.shape}")

### 1. Removing Redundant Columns
**Why?** The `Unnamed: 0` column is an artifact of saving/loading CSVs without specifying `index=False` in previous iterations. It provides no analytical value.

**Action:** Drop `Unnamed: 0` if it exists.

In [ ]:
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)
print(f"Columns after dropping index: {df.columns.tolist()}")

### 2. Handling Missing Values & Advanced Imputation
**Why?** Missing values in categorical columns like `publisher` or `primary_genre` can disrupt grouping operations. Furthermore, missing `review_percentage` values can be mathematically recovered if review counts are present.

**Actions:** 
1. Fill missing `publisher` and `developer` with 'Unknown'.
2. Fill missing `primary_genre` with 'Uncategorized'.
3. Calculate `review_percentage` where missing using `positive_reviews` and `total_reviews`.

In [ ]:
# Filling categorical blanks
df['publisher'] = df['publisher'].fillna('Unknown')
df['developer'] = df['developer'].fillna('Unknown')
df['primary_genre'] = df['primary_genre'].fillna('Uncategorized')

# Recalculating review percentage
# Formula: (Positive / Total) * 100
mask = df['review_percentage'].isna() & (df['total_reviews'] > 0)
df.loc[mask, 'review_percentage'] = (df.loc[mask, 'positive_reviews'] / df.loc[mask, 'total_reviews']) * 100

print("Advanced imputation complete.")

### 3. Extracting AppID from Link
**Why?** The `link` column contains the unique Steam Application ID. Extracting this as a numeric ID makes it easier to reference and join with other datasets.

**Action:** Use regex to extract the numeric ID.

In [ ]:
df['appid'] = df['link'].str.extract(r'/app/(\d+)/')
df['appid'] = pd.to_numeric(df['appid'], errors='coerce')
print("AppIDs extracted.")

### 4. Correcting Data Types (Datetime)
**Why?** String dates limit temporal analysis.

**Action:** Convert `release` and `all_time_peak_date` to Datetime objects.

In [ ]:
df['release'] = pd.to_datetime(df['release'], errors='coerce')
df['all_time_peak_date'] = pd.to_datetime(df['all_time_peak_date'], errors='coerce')
print("Date columns standardized.")

### 5. Cleaning Numeric Columns
**Why?** Commas in numbers prevent mathematical calculations.

**Action:** Replace commas and convert to integers.

In [ ]:
cols_to_clean = ['players_right_now', '24_hour_peak', 'all_time_peak']
for col in cols_to_clean:
    if df[col].dtype == 'object':
        df[col] = df[col].str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print("Numeric conversion complete.")

### 6. Cleaning Genre Strings
**Why?** Internal IDs in parentheses clutter the text labels.

**Action:** Strip content within parentheses.

In [ ]:
def clean_genre(val):
    if pd.isna(val): return val
    return re.sub(r'\s*\(\d+\)', '', str(val))

df['primary_genre'] = df['primary_genre'].apply(clean_genre)
df['store_genres'] = df['store_genres'].apply(clean_genre)
print("Genre labels normalized.")

### 7. Duplicate Removal
**Why?** Duplicates bias analysis results.

**Action:** Drop duplicates based on `link`.

In [ ]:
df.drop_duplicates(subset=['link'], keep='first', inplace=True)
print(f"Final shape after deduplication: {df.shape}")

### 8. Final Export
**Action:** Save to `data/processed/cleaned_game_data.csv`.

In [ ]:
processed_path = '../data/processed/'
os.makedirs(processed_path, exist_ok=True)
output_file = os.path.join(processed_path, 'cleaned_game_data.csv')
df.to_csv(output_file, index=False)
print(f"Perfectly standardized data saved to: {output_file}")

In [13]:
df

,game,link,release,peak_players,positive_reviews,negative_reviews,total_reviews,rating,primary_genre,store_genres,publisher,developer,detected_technologies,store_asset_mod_time,review_percentage,players_right_now,24_hour_peak,all_time_peak,all_time_peak_date,appid
0,ASCENXION,/app/1000000/,2021-05-14,9,26,6,32,70.34,Indie,"Action, Adventure, Indie",PsychoFlux Entertainment,IndigoBlue Game Studio,Engine.GameMaker,2021-05-14,81.0,0,1,9,2021-05-15,1000000
1,Crown Trick,/app/1000010/,2020-10-16,6594,4186,659,4845,83.57,RPG,"Adventure, Indie, RPG, Strategy",Team17,NEXT Studios,Engine.Unity; SDK.Wwise,2021-08-23,86.0,23,39,6594,2020-10-19,1000010
2,Cook Serve Delicious! 3?!,/app/1000030/,2020-10-14,491,1678,130,1808,88.33,Action,"Action, Indie, Simulation, Strategy",Vertigo Gaming Inc.,Vertigo Gaming Inc.,Engine.GameMaker,2022-08-10,92.0,30,56,491,2020-01-29,1000030
3,细胞战争,/app/1000040/,2019-03-30,1,0,1,1,40.58,Action,"Action, Casual, Indie, Simulation",DoubleC Games,DoubleC Games,Engine.Unity,2019-03-31,NaN,0,0,1,2019-03-31,1000040
4,Zengeon,/app/1000080/,2019-06-24,469,1010,474,1484,66.06,Indie,"Action, Adventure, Indie, RPG",2P Games,IndieLeague Studio,Engine.Unity,2022-10-29,68.0,1,4,469,2019-06-25,1000080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67566,Becalm,/app/999830/,2019-01-17,16,291,55,346,78.24,Indie,"Casual, Free to Play, Indie",Colorfiction,Colorfiction,Engine.Unity,2020-01-10,84.0,0,3,16,2019-01-19,999830
67567,Enemy On Board,/app/999860/,2020-05-08,891,1142,425,1567,70.38,Casual,"Action, Casual, Free to Play, Indie, Early Access",Windwalk Games,Windwalk Games,Engine.Unity; SDK.Discord; SDK.UnityURP,2020-10-01,72.0,0,7,891,2020-05-10,999860
67568,Bruken,/app/999890/,2019-04-11,1,2,0,2,64.08,Casual,"Adventure, Casual, Indie",FlairBot Games,FlairBot Games,Engine.Unity,2019-05-02,NaN,0,0,1,2019-04-13,999890
67569,Fantasy Sino-Japanese War 幻想甲午,/app/999930/,2019-01-11,2,2,2,4,50.00,RPG,"Indie, RPG, Strategy, Early Access",张八万工作室,张八万工作室,NaN,2018-12-19,NaN,0,0,2,2019-01-12,999930
